In [ ]:
# import os
# from pathlib import Path
# import datetime
# from tqdm import tqdm
# from dataclasses import dataclass, asdict
# import polars as pl 
# import numpy as np
# from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
# from sklearn.preprocessing import StandardScaler
# import kaggle_evaluation.default_inference_server

# import pandas as pd
# from typing import List, Dict, Tuple
# from lightgbm import LGBMRegressor          # LightGBM regression model
# from sklearn.metrics import r2_score        # Official evaluation metric: R-squared
# import warnings
# import math
# warnings.filterwarnings("ignore", category=RuntimeWarning)

# # ===========================
# # Define constants
# # ===========================
# EPS = 1e-8  # Prevent division by zero
# MAX_POSITION = 2
# MIN_POSITION = 0
# ANNUAL_DAYS = 252
# MIN_SIGNAL = 0.0
# MAX_SIGNAL = 2.0

# # Main parameter for the simple signal mapping

# SIGNAL_MULTIPLIER = 100.0

# DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')

# # ===========================
# # Strategy scoring function: the competition Sharpe variant
# # ===========================
# def adjusted_sharpe(solution: pd.DataFrame, submission: pd.DataFrame) -> float:
#     """
#     Calculate a penalized Sharpe-like score:
#     - solution: observed data with forward_returns and risk_free_rate
#     - submission: predicted positions in the prediction column, in [0,2]
#     """
#     sol = solution.copy().reset_index(drop=False)
#     sub = submission.copy().reset_index(drop=False)
#     if sol.shape[0] != sub.shape[0]:
#         raise ValueError("solution and submission have different lengths")
#     df = sol.copy()
#     df['position'] = sub['prediction'].astype(float).values
    
#     # Check the position range
#     if df['position'].max() > MAX_POSITION + 1e-9:
#         raise ValueError(f"Position exceeds the upper bound {MAX_POSITION}")
#     if df['position'].min() < MIN_POSITION - 1e-9:
#         raise ValueError(f"Position is below the lower bound {MIN_POSITION}")
    
#     # Strategy return: risk-free return plus position times excess return
#     df['strategy_returns'] = (
#         df['risk_free_rate'] * (1 - df['position']) +
#         df['position'] * df['forward_returns']
#     )
#     excess = df['strategy_returns'] - df['risk_free_rate']
#     cum_excess = float((1.0 + excess).prod())
#     mean_excess = (
#         cum_excess ** (1.0 / len(df)) - 1.0
#         if cum_excess > 0 else -1.0
#     )
#     std_excess = float(df['strategy_returns'].std())
    
#     if std_excess == 0:
#         return 0.0
    
#     # Base Sharpe ratio
#     sharpe = mean_excess / (std_excess + EPS) * math.sqrt(ANNUAL_DAYS)
#     strat_vol = float(std_excess * math.sqrt(ANNUAL_DAYS) * 100.0)
    
#     # Market benchmark
#     market_excess = df['forward_returns'] - df['risk_free_rate']
#     market_cum = float((1.0 + market_excess).prod())
#     market_mean = (
#         market_cum ** (1.0 / len(df)) - 1.0
#         if market_cum > 0 else -1.0
#     )
#     market_std = float(df['forward_returns'].std())
#     market_vol = float(market_std * math.sqrt(ANNUAL_DAYS) * 100.0)
    
#     # Penalty for excessive volatility
#     excess_vol_penalty = (
#         1.0 + max(0.0, strat_vol / (market_vol + EPS) - 1.2)
#         if market_vol > 0 else 1.0
#     )
#     # Penalty for insufficient returns
#     return_gap = max(0.0, (market_mean - mean_excess) * 100.0 * ANNUAL_DAYS)
#     return_penalty = 1.0 + (return_gap ** 2) / 100.0
    
#     score = sharpe / (excess_vol_penalty * return_penalty + EPS)
#     return float(min(score, 1_000_000.0))

# # ===========================
# # Calculate RSI
# # ===========================
# def compute_RSI_from_prices(price_series, window=14):
#     """
#     Calculate RSI from a price series:
#     - Calculate price gains and losses
#     - Apply EWM smoothing
#     """
#     delta = price_series.diff()
#     up = delta.clip(lower=0.0)
#     down = -delta.clip(upper=0.0)
#     roll_up = up.ewm(alpha=1/window, adjust=False).mean()
#     roll_down = down.ewm(alpha=1/window, adjust=False).mean()
#     rs = roll_up / (roll_down + EPS)
#     return 100 - 100 / (1 + rs).fillna(50.0)

# # ===========================
# # Shared train/test feature engineering (leakage-free version)
# # ===========================
# def preprocess_and_make_features(df, windows=[5, 10, 21, 42, 63, 126]):
#     """
#     Apply the same preprocessing to the supplied df:
#     - Sort by date_id
#     - Drop market_forward_excess_returns to avoid leakage
#     - Construct U1 and U2
#     - Impute numeric columns with EWM using past data only
#     - Construct time-series indicators from lagged_forward_returns
#       Do not use forward_returns directly; use lagged returns and technical indicators
#     """
#     df = df.sort_values("date_id").reset_index(drop=True)

#     # Present only in train: avoid using the target as a feature
#     if "market_forward_excess_returns" in df.columns:
#         df = df.drop(columns=["market_forward_excess_returns"])

#     # Construct U1 and U2 when the required columns exist
#     if {"I2", "I1"}.issubset(df.columns):
#         df["U1"] = df["I2"] - df["I1"]
#     if {"M11", "I2", "I9", "I7"}.issubset(df.columns):
#         df["U2"] = df["M11"] / ((df["I2"] + df["I9"] + df["I7"]) / 3.0)

#     # Replace inf and -inf with NaN
#     df = df.replace([np.inf, -np.inf], np.nan)

#     # EWM imputation for numeric columns, moving forward without future information
#     numeric_cols = df.select_dtypes(include=[np.number]).columns
#     for col in numeric_cols:
#         df[col] = df[col].fillna(df[col].ewm(com=0.5, min_periods=1).mean())

#     # ========= Technical indicators from lagged_forward_returns =========
#     # Construct lagged_forward_returns for train
#     # The test set provides lagged_forward_returns
#     if "lagged_forward_returns" not in df.columns:
#         raise ValueError("df lacks lagged_forward_returns; construct it first or load it from test.csv")

#     lagged = df["lagged_forward_returns"]
#     price = (1.0 + lagged.fillna(0.0)).cumprod()

#     new_cols = {}

#     # Lagged returns
#     new_cols["ret_lag_1"] = lagged.fillna(0.0)
#     for lag in [2, 3, 5, 10]:
#         new_cols[f"ret_lag_{lag}"] = lagged.shift(lag - 1).fillna(0.0)

#     # Rolling indicators
#     for w in windows:
#         new_cols[f"mom_{w}"] = price.pct_change(w).fillna(0.0)
#         new_cols[f"roll_mean_{w}"] = lagged.rolling(w, min_periods=1).mean()
#         new_cols[f"roll_std_{w}"] = lagged.rolling(w, min_periods=1).std().fillna(0.0)
#         new_cols[f"roll_skew_{w}"] = lagged.rolling(w, min_periods=5).skew().fillna(0.0)
#         ma = price.rolling(w, min_periods=1).mean()
#         new_cols[f"bias_ma_{w}"] = (price - ma) / (ma + EPS)

#     # Other technical indicators
#     new_cols["skew_21"] = lagged.rolling(21, min_periods=5).skew().fillna(0.0)
#     new_cols["rsi_14"] = compute_RSI_from_prices(price, window=14)

#     ema12 = price.ewm(span=12, adjust=False).mean()
#     ema26 = price.ewm(span=26, adjust=False).mean()
#     macd = ema12 - ema26
#     macd_signal = macd.ewm(span=9, adjust=False).mean()

#     new_cols["macd"] = macd.fillna(0.0)
#     new_cols["macd_signal"] = macd_signal.fillna(0.0)
#     new_cols["macd_hist"] = (macd - macd_signal).fillna(0.0)

#     new_cols["vol_21"] = lagged.rolling(21, min_periods=1).std().fillna(0.0)
#     new_cols["vol_63"] = lagged.rolling(63, min_periods=1).std().fillna(0.0)
#     new_cols["vol_ratio_63_21"] = new_cols["vol_63"] / (new_cols["vol_21"] + EPS)

#     # new_cols["mom21_over_vol21"] = new_cols["mom_21"] / (new_cols["vol_21"] + EPS)
#     new_cols["macd_x_rsi"] = new_cols["macd"] * (new_cols["rsi_14"] - 50)
#     # new_cols["bias_x_mom"] = new_cols["bias_ma_21"] * new_cols["mom_21"]
#     new_cols["skew_x_vol"] = new_cols["skew_21"] * new_cols["vol_21"]

#     # Previous-day return with an explicit name
#     new_cols["forward_return_prev"] = lagged.fillna(0.0)

#     new_df = pd.DataFrame(new_cols)
#     df = pd.concat([df.reset_index(drop=True), new_df], axis=1)
#     df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

#     return df

# # ===========================
# # Time-series cross-validation for local evaluation
# # ===========================
# def time_series_cv_lgbm(
#     train_df,
#     train_len=2000,
#     val_len=180,
#     step=500,
#     skip_initial=1300,
# ):
#     """
#     Rolling time-series cross-validation with LightGBM:
#     - Automatically call preprocess_and_make_features
#     - Discard the first skip_initial rows, then roll the windows:
#         train : [i : i+train_len)
#         val   : [i+train_len : i+train_len+val_len)
#     - Print index ranges, date_id ranges, R-squared, and Sharpe for each fold
#     """
#     # Apply the shared feature engineering first
#     train_df = preprocess_and_make_features(train_df)
#     print("Feature engineering complete, shape:", train_df.shape)

#     # Exclude time and labels (forward_returns, risk_free_rate) from features
#     feature_cols = [
#         c for c in train_df.columns
#         if c not in ["date_id", "forward_returns", "risk_free_rate"]
#     ]

#     # LightGBM parameters
#     lgbm_params = dict(
#         n_estimators=1000,
#         learning_rate=0.001,
#         num_leaves=64,
#         max_depth=5,
#         random_state=42,
#         n_jobs=-1,
#         verbose=-1,
#     )

#     # Discard the initial noisy segment
#     train_df = train_df.iloc[skip_initial:].reset_index(drop=True)
#     n = len(train_df)
#     print("Data shape used for CV:", train_df.shape)

#     scores = []
#     sharpe_scores = []
#     fold_info = []
#     fold_idx = 0
#     start_train = 0

#     # Rolling-window cross-validation
#     while True:
#         end_train = start_train + train_len
#         start_val = end_train
#         end_val = start_val + val_len
#         if end_val > n:
#             break

#         df_train_fold = train_df.iloc[start_train:end_train]
#         df_val_fold = train_df.iloc[start_val:end_val]

#         X_train = df_train_fold[feature_cols].values
#         y_train = df_train_fold["forward_returns"].values
#         X_val = df_val_fold[feature_cols].values
#         y_val = df_val_fold["forward_returns"].values

#         model = LGBMRegressor(**lgbm_params)
#         model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="l2")

#         # R²
#         y_pred = model.predict(X_val)
#         r2 = r2_score(y_val, y_pred)

#         # Strategy Sharpe ratio
#         position_raw = 1.0 + SIGNAL_MULTIPLIER * y_pred
#         position = np.clip(position_raw, MIN_SIGNAL, MAX_SIGNAL)
#         submission = pd.DataFrame({"prediction": position}, index=df_val_fold.index)

#         solution = df_val_fold[["forward_returns", "risk_free_rate"]].copy()
#         sharpe_score = adjusted_sharpe(solution, submission)

#         scores.append(r2)
#         sharpe_scores.append(sharpe_score)

#         train_date_start = df_train_fold["date_id"].iloc[0]
#         train_date_end   = df_train_fold["date_id"].iloc[-1]
#         val_date_start   = df_val_fold["date_id"].iloc[0]
#         val_date_end     = df_val_fold["date_id"].iloc[-1]

#         fold_info.append({
#             "fold": fold_idx,
#             "train_range_idx": (start_train, end_train),
#             "val_range_idx": (start_val, end_val),
#             "train_date_range": (int(train_date_start), int(train_date_end)),
#             "val_date_range": (int(val_date_start), int(val_date_end)),
#             "r2": float(r2),
#             "sharpe": float(sharpe_score),
#         })

#         print(
#             f"Fold {fold_idx}: "
#             f"train_idx=[{start_train}:{end_train}), "
#             f"val_idx=[{start_val}:{end_val}), "
#             f"train_date=[{train_date_start} ~ {train_date_end}], "
#             f"val_date=[{val_date_start} ~ {val_date_end}], "
#             f"R2={r2:.6f}, Sharpe={sharpe_score:.6f}"
#         )

#         start_train += step
#         fold_idx += 1

#     print("=" * 70)
#     print(f"Total folds: {fold_idx}")
#     print("R2 scores:", [float(s) for s in scores])
#     print("Sharpe scores:", [float(s) for s in sharpe_scores])

#     if scores:
#         print(f"Mean R2:   {np.mean(scores):.6f}")
#         print(f"Std  R2:   {np.std(scores):.6f}")
#     if sharpe_scores:
#         print(f"Mean Sharpe: {np.mean(sharpe_scores):.6f}")
#         print(f"Std  Sharpe: {np.std(sharpe_scores):.6f}")
#     print("=" * 70)

#     return scores, fold_info















# # ===========================
# # Read train, construct lagged features, and run local CV
# # ===========================
# train_raw = pd.read_csv("/kaggle/input/hull-tactical-market-prediction/train.csv")
# train_raw = train_raw.sort_values("date_id").reset_index(drop=True)

# # Construct 3 lagged features in train to match the test structure
# train_raw['lagged_forward_returns'] = train_raw["forward_returns"].shift(1)
# train_raw['lagged_risk_free_rate'] = train_raw["risk_free_rate"].shift(1)
# train_raw['lagged_market_forward_excess_returns'] = train_raw["market_forward_excess_returns"].shift(1)

# print("Original training data shape:", train_raw.shape)

# # Optional local cross-validation
# scores, fold_info = time_series_cv_lgbm(
#     train_df=train_raw,
#     train_len=5000,
#     val_len=180,
#     step=500,
#     skip_initial=1300,
# )

# # ===========================
# # Train the final submission model on rows 1300:-180
# # ===========================
# # 1) Apply the same feature engineering
# train_fe = preprocess_and_make_features(train_raw)

# # 2) Select the 1300:-180 training interval
# train_fe_final = train_fe.iloc[1300:-180].reset_index(drop=True)
# print("Data shape for final model training:", train_fe_final.shape)

# # 3) Define feature columns consistently with CV
# feature_cols_final = [
#     c for c in train_fe_final.columns
#     if c not in ["date_id", "forward_returns", "risk_free_rate"]
# ]

# X_final = train_fe_final[feature_cols_final].values
# y_final = train_fe_final["forward_returns"].values

# # Support test-set imputation during predict
# train_feature_medians_final = train_fe_final[feature_cols_final].median()

# lgbm_params_final = dict(
#     n_estimators=5000,
#     learning_rate=0.001,
#     num_leaves=64,
#     max_depth=5,
#     random_state=42,
#     n_jobs=-1,
#     verbose=-1,
# )

# model_final = LGBMRegressor(**lgbm_params_final)
# model_final.fit(X_final, y_final)
# print("Final LightGBM model trained; feature count:", len(feature_cols_final))

# # ===========================
# # Kaggle evaluation interface: predict(test: pl.DataFrame)
# # Use exactly the same feature engineering as training, in streaming mode
# # ===========================

# # Global variable to accumulate test history during evaluation
# test_history_df = None

# def predict(test: pl.DataFrame) -> float:
#     """
#     Prediction function called by Kaggle default_inference_server:
#     - Input: a Polars DataFrame, usually one row at a time in chronological order
#     - Output: allocation for the latest row, in [0, 2]

#     Key points:
#     - Use the same preprocess_and_make_features as training
#     - Accumulate test rows in test_history_df so rolling/EWM operations use past data only
#     """
#     global test_history_df, model_final, feature_cols_final, train_feature_medians_final

#     # pl → pandas
#     test_pd = test.to_pandas()

#     # Append to the history, assuming calls arrive in date order
#     if test_history_df is None:
#         test_history_df = test_pd.copy()
#     else:
#         test_history_df = pd.concat([test_history_df, test_pd], axis=0, ignore_index=True)

#     # test already contains lagged_forward_returns / lagged_risk_free_rate / lagged_market_forward_excess_returns
#     # Reuse the shared feature engineering function
#     test_fe = preprocess_and_make_features(test_history_df)

#     # Add any missing training feature columns using their training medians
#     for col in feature_cols_final:
#         if col not in test_fe.columns:
#             default_val = train_feature_medians_final.get(col, 0.0)
#             test_fe[col] = default_val

#     X_all = test_fe[feature_cols_final].copy()
#     X_all = X_all.replace([np.inf, -np.inf], np.nan)
#     X_all = X_all.fillna(train_feature_medians_final)

#     # Use the latest row as the current feature vector
#     X_last = X_all.iloc[[-1]].values
#     pred_ret = float(model_final.predict(X_last)[0])

#     # Map to a position allocation
#     position_raw = 1.0 + SIGNAL_MULTIPLIER * pred_ret
#     alloc = float(np.clip(position_raw, MIN_SIGNAL, MAX_SIGNAL))
#     return alloc

In [ ]:
import os
from pathlib import Path
import datetime
from tqdm import tqdm
from dataclasses import dataclass, asdict
import polars as pl 
import numpy as np
from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
import kaggle_evaluation.default_inference_server

import pandas as pd
from typing import List, Dict, Tuple
from lightgbm import LGBMRegressor          # LightGBM regression model
from sklearn.metrics import r2_score        # Official evaluation metric: R-squared
import warnings
import math
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ===========================
# Define constants
# ===========================
EPS = 1e-8  # Prevent division by zero
MAX_POSITION = 2
MIN_POSITION = 0
ANNUAL_DAYS = 252
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

# Main parameter for the simple signal mapping, overwritten after grid search
SIGNAL_MULTIPLIER = 100.0

DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')

# ===========================
# Strategy scoring function: the competition Sharpe variant
# ===========================
def adjusted_sharpe(solution: pd.DataFrame, submission: pd.DataFrame) -> float:
    """
    Calculate a penalized Sharpe-like score:
    - solution: observed data with forward_returns and risk_free_rate.
    - submission: predicted positions in the prediction column, in [0,2].
    """
    sol = solution.copy().reset_index(drop=False)
    sub = submission.copy().reset_index(drop=False)
    if sol.shape[0] != sub.shape[0]:
        raise ValueError("solution and submission have different lengths")
    df = sol.copy()
    df['position'] = sub['prediction'].astype(float).values
    
    # Check the position range
    if df['position'].max() > MAX_POSITION + 1e-9:
        raise ValueError(f"Position exceeds the upper bound {MAX_POSITION}")
    if df['position'].min() < MIN_POSITION - 1e-9:
        raise ValueError(f"Position is below the lower bound {MIN_POSITION}")
    
    # Strategy return: risk-free return plus position times excess return
    df['strategy_returns'] = (
        df['risk_free_rate'] * (1 - df['position']) +
        df['position'] * df['forward_returns']
    )
    excess = df['strategy_returns'] - df['risk_free_rate']
    cum_excess = float((1.0 + excess).prod())
    mean_excess = (
        cum_excess ** (1.0 / len(df)) - 1.0
        if cum_excess > 0 else -1.0
    )
    std_excess = float(df['strategy_returns'].std())
    
    if std_excess == 0:
        return 0.0
    
    # Base Sharpe ratio
    sharpe = mean_excess / (std_excess + EPS) * math.sqrt(ANNUAL_DAYS)
    strat_vol = float(std_excess * math.sqrt(ANNUAL_DAYS) * 100.0)
    
    # Market benchmark
    market_excess = df['forward_returns'] - df['risk_free_rate']
    market_cum = float((1.0 + market_excess).prod())
    market_mean = (
        market_cum ** (1.0 / len(df)) - 1.0
        if market_cum > 0 else -1.0
    )
    market_std = float(df['forward_returns'].std())
    market_vol = float(market_std * math.sqrt(ANNUAL_DAYS) * 100.0)
    
    # Penalty for excessive volatility
    excess_vol_penalty = (
        1.0 + max(0.0, strat_vol / (market_vol + EPS) - 1.2)
        if market_vol > 0 else 1.0
    )
    # Penalty for insufficient returns
    return_gap = max(0.0, (market_mean - mean_excess) * 100.0 * ANNUAL_DAYS)
    return_penalty = 1.0 + (return_gap ** 2) / 100.0
    
    score = sharpe / (excess_vol_penalty * return_penalty + EPS)
    return float(min(score, 1_000_000.0))

# ===========================
# Calculate RSI
# ===========================
def compute_RSI_from_prices(price_series, window=14):
    """
    Calculate RSI from a price series:
    - Calculate price gains and losses.
    - Apply EWM smoothing.
    """
    delta = price_series.diff()
    up = delta.clip(lower=0.0)
    down = -delta.clip(upper=0.0)
    roll_up = up.ewm(alpha=1/window, adjust=False).mean()
    roll_down = down.ewm(alpha=1/window, adjust=False).mean()
    rs = roll_up / (roll_down + EPS)
    return 100 - 100 / (1 + rs).fillna(50.0)

# ===========================
# Shared train/test feature engineering (leakage-free version)
# ===========================
def preprocess_and_make_features(df, windows=[5, 10, 21, 42, 63, 126]):
    """
    Apply the same preprocessing to the supplied df:
    - Sort by date_id.
    - Drop market_forward_excess_returns to avoid leakage.
    - Construct U1 and U2.
    - Impute numeric columns with EWM using past data only.
    - Construct time-series indicators from lagged_forward_returns.
      Do not use forward_returns directly; use lagged returns and technical indicators.
    """
    df = df.sort_values("date_id").reset_index(drop=True)

    # Present only in train: avoid using the target as a feature
    if "market_forward_excess_returns" in df.columns:
        df = df.drop(columns=["market_forward_excess_returns"])

    # Construct U1 and U2 when the required columns exist
    if {"I2", "I1"}.issubset(df.columns):
        df["U1"] = df["I2"] - df["I1"]
    if {"M11", "I2", "I9", "I7"}.issubset(df.columns):
        df["U2"] = df["M11"] / ((df["I2"] + df["I9"] + df["I7"]) / 3.0)

    # Replace inf and -inf with NaN
    df = df.replace([np.inf, -np.inf], np.nan)

    # EWM imputation for numeric columns, moving forward without future information
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        # Impute missing values with an exponentially weighted mean using historical data only
        df[col] = df[col].fillna(df[col].ewm(com=0.5, min_periods=1).mean())

    # ========= Technical indicators from lagged_forward_returns =========
    # Construct lagged_forward_returns for train
    # The test set provides lagged_forward_returns
    if "lagged_forward_returns" not in df.columns:
        raise ValueError("df lacks lagged_forward_returns; construct it first or load it from test.csv")

    lagged = df["lagged_forward_returns"]
    # Compound lagged returns to construct a synthetic price series
    price = (1.0 + lagged.fillna(0.0)).cumprod()

    new_cols = {}

    # Lagged return features
    new_cols["ret_lag_1"] = lagged.fillna(0.0)
    for lag in [2, 3, 5, 10]:
        new_cols[f"ret_lag_{lag}"] = lagged.shift(lag - 1).fillna(0.0)

    # Rolling summary statistics
    for w in windows:
        new_cols[f"mom_{w}"] = price.pct_change(w).fillna(0.0)
        new_cols[f"roll_mean_{w}"] = lagged.rolling(w, min_periods=1).mean()
        new_cols[f"roll_std_{w}"] = lagged.rolling(w, min_periods=1).std().fillna(0.0)
        new_cols[f"roll_skew_{w}"] = lagged.rolling(w, min_periods=5).skew().fillna(0.0)
        ma = price.rolling(w, min_periods=1).mean()
        new_cols[f"bias_ma_{w}"] = (price - ma) / (ma + EPS)

    # Other technical indicators
    new_cols["skew_21"] = lagged.rolling(21, min_periods=5).skew().fillna(0.0)
    new_cols["rsi_14"] = compute_RSI_from_prices(price, window=14)

    ema12 = price.ewm(span=12, adjust=False).mean()
    ema26 = price.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False).mean()

    new_cols["macd"] = macd.fillna(0.0)
    new_cols["macd_signal"] = macd_signal.fillna(0.0)
    new_cols["macd_hist"] = (macd - macd_signal).fillna(0.0)

    new_cols["vol_21"] = lagged.rolling(21, min_periods=1).std().fillna(0.0)
    new_cols["vol_63"] = lagged.rolling(63, min_periods=1).std().fillna(0.0)
    new_cols["vol_ratio_63_21"] = new_cols["vol_63"] / (new_cols["vol_21"] + EPS)

    # Interaction features
    new_cols["macd_x_rsi"] = new_cols["macd"] * (new_cols["rsi_14"] - 50)
    new_cols["skew_x_vol"] = new_cols["skew_21"] * new_cols["vol_21"]

    # Previous-day return with an explicit name
    new_cols["forward_return_prev"] = lagged.fillna(0.0)

    # Merge new features back into the original DataFrame
    new_df = pd.DataFrame(new_cols)
    df = pd.concat([df.reset_index(drop=True), new_df], axis=1)
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return df

# ===========================
# Time-series cross-validation for local evaluation
# ===========================
def time_series_cv_lgbm(
    train_df,
    train_len=2000,
    val_len=180,
    step=500,
    skip_initial=1300,
    signal_multiplier=None,   # New: allow different SIGNAL_MULTIPLIER values
):
    """
    Rolling time-series cross-validation with LightGBM:
    - Automatically call preprocess_and_make_features.
    - Discard the first skip_initial rows, then roll the windows:
        train : [i : i+train_len)
        val   : [i+train_len : i+train_len+val_len)
    - Print index ranges, date_id ranges, R-squared, and Sharpe for each fold.
    - signal_multiplier: position scaling factor; use global SIGNAL_MULTIPLIER if None.
    """
    # Use the current global SIGNAL_MULTIPLIER if unspecified
    if signal_multiplier is None:
        signal_multiplier = SIGNAL_MULTIPLIER

    # Apply the shared feature engineering first
    train_df = preprocess_and_make_features(train_df)
    print("Feature engineering complete, shape:", train_df.shape)

    # Exclude time and labels (forward_returns, risk_free_rate) from features
    feature_cols = [
        c for c in train_df.columns
        if c not in ["date_id", "forward_returns", "risk_free_rate"]
    ]

    # LightGBM parameters
    lgbm_params = dict(
        n_estimators=1000,
        learning_rate=0.001,
        num_leaves=64,
        max_depth=5,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    # Discard the initial noisy segment
    train_df = train_df.iloc[skip_initial:].reset_index(drop=True)
    n = len(train_df)
    print("Data shape used for CV:", train_df.shape)

    scores = []          # Store each fold's R-squared
    sharpe_scores = []   # Store each fold's Sharpe ratio
    fold_info = []       # Save detailed fold information
    fold_idx = 0
    start_train = 0

    # Rolling-window cross-validation
    while True:
        end_train = start_train + train_len
        start_val = end_train
        end_val = start_val + val_len
        if end_val > n:
            break

        df_train_fold = train_df.iloc[start_train:end_train]
        df_val_fold = train_df.iloc[start_val:end_val]

        X_train = df_train_fold[feature_cols].values
        y_train = df_train_fold["forward_returns"].values
        X_val = df_val_fold[feature_cols].values
        y_val = df_val_fold["forward_returns"].values

        model = LGBMRegressor(**lgbm_params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="l2")

        # R²
        y_pred = model.predict(X_val)
        r2 = r2_score(y_val, y_pred)

        # Strategy Sharpe ratio using the current signal_multiplier
        position_raw = 1.0 + signal_multiplier * y_pred
        position = np.clip(position_raw, MIN_SIGNAL, MAX_SIGNAL)
        submission = pd.DataFrame({"prediction": position}, index=df_val_fold.index)

        solution = df_val_fold[["forward_returns", "risk_free_rate"]].copy()
        sharpe_score = adjusted_sharpe(solution, submission)

        scores.append(r2)
        sharpe_scores.append(sharpe_score)

        train_date_start = df_train_fold["date_id"].iloc[0]
        train_date_end   = df_train_fold["date_id"].iloc[-1]
        val_date_start   = df_val_fold["date_id"].iloc[0]
        val_date_end     = df_val_fold["date_id"].iloc[-1]

        fold_info.append({
            "fold": fold_idx,
            "train_range_idx": (start_train, end_train),
            "val_range_idx": (start_val, end_val),
            "train_date_range": (int(train_date_start), int(train_date_end)),
            "val_date_range": (int(val_date_start), int(val_date_end)),
            "r2": float(r2),
            "sharpe": float(sharpe_score),
        })

        print(
            f"Fold {fold_idx}: "
            f"train_idx=[{start_train}:{end_train}), "
            f"val_idx=[{start_val}:{end_val}), "
            f"train_date=[{train_date_start} ~ {train_date_end}], "
            f"val_date=[{val_date_start} ~ {val_date_end}], "
            f"R2={r2:.6f}, Sharpe={sharpe_score:.6f}, "
            f"signal_multiplier={signal_multiplier}"
        )

        start_train += step
        fold_idx += 1

    print("=" * 70)
    print(f"Total folds: {fold_idx}")
    print("R2 scores:", [float(s) for s in scores])
    print("Sharpe scores:", [float(s) for s in sharpe_scores])

    if scores:
        print(f"Mean R2:   {np.mean(scores):.6f}")
        print(f"Std  R2:   {np.std(scores):.6f}")
    if sharpe_scores:
        print(f"Mean Sharpe: {np.mean(sharpe_scores):.6f}")
        print(f"Std  Sharpe: {np.std(sharpe_scores):.6f}")
    print("=" * 70)

    # Return the R-squared list, Sharpe list, and detailed fold information
    return scores, sharpe_scores, fold_info

# ===========================
# Read train and construct lagged features
# ===========================
train_raw = pd.read_csv("/kaggle/input/hull-tactical-market-prediction/train.csv")
train_raw = train_raw.sort_values("date_id").reset_index(drop=True)

# Construct 3 lagged features in train to match the test structure
train_raw['lagged_forward_returns'] = train_raw["forward_returns"].shift(1)
train_raw['lagged_risk_free_rate'] = train_raw["risk_free_rate"].shift(1)
train_raw['lagged_market_forward_excess_returns'] = train_raw["market_forward_excess_returns"].shift(1)

print("Original training data shape:", train_raw.shape)

# ===========================
# Grid search over train_len and SIGNAL_MULTIPLIER
# ===========================
# Candidate training-window lengths
train_len_candidates = [1000, 2000, 3000, 4000, 5000]
# Candidate position scaling factors
signal_multiplier_candidates = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]

best_sharpe_overall = -np.inf           # Global best Sharpe ratio
best_train_len = None                   # Best train_len
best_signal_multiplier = None           # Best SIGNAL_MULTIPLIER

# To avoid repeating feature engineering excessively within time_series_cv_lgbm,
# it could be done in advance; here we pass raw train_raw to stay consistent with the function's internal preprocessing.
for tl in train_len_candidates:
    for sm in signal_multiplier_candidates:
        print(f"\n=== Starting CV: train_len={tl}, SIGNAL_MULTIPLIER={sm} ===")
        # Run time-series CV and return R-squared, Sharpe, and fold details
        scores, sharpe_scores, fold_info = time_series_cv_lgbm(
            train_df=train_raw,
            train_len=tl,
            val_len=180,
            step=500,
            skip_initial=1300,
            signal_multiplier=sm,
        )

        # Skip parameter combinations without valid folds
        if not sharpe_scores:
            continue

        mean_sharpe = float(np.mean(sharpe_scores))
        print(f"Parameter combination train_len={tl}, SIGNAL_MULTIPLIER={sm} has mean Sharpe={mean_sharpe:.6f}")

        # Update the global best
        if mean_sharpe > best_sharpe_overall:
            best_sharpe_overall = mean_sharpe
            best_train_len = tl
            best_signal_multiplier = sm
            print(f"*** New best combination: train_len={tl}, SIGNAL_MULTIPLIER={sm}, mean Sharpe={mean_sharpe:.6f} ***")

print("\n" + "#" * 80)
print("Grid search complete!")
print(f"Best parameter combination: train_len={best_train_len}, SIGNAL_MULTIPLIER={best_signal_multiplier}, mean Sharpe={best_sharpe_overall:.6f}")
print("#" * 80 + "\n")

# Update the global SIGNAL_MULTIPLIER with the best value for later training and predict
SIGNAL_MULTIPLIER = float(best_signal_multiplier)

# ===========================
# Train the final submission model using the best train_len
# ===========================
# 1) Apply the same feature engineering
train_fe = preprocess_and_make_features(train_raw)

# Use the same initial exclusion and final holdout settings as CV
skip_initial_final = 1300   # Match skip_initial in CV
val_len_final = 180         # Match val_len in CV, excluding the final rows

# Remove the first skip_initial and last val_len rows after feature engineering
train_window = train_fe.iloc[skip_initial_final:-val_len_final].reset_index(drop=True)

# Train on the last best_train_len rows of this interval
if best_train_len is not None and best_train_len < len(train_window):
    train_fe_final = train_window.iloc[-best_train_len:].reset_index(drop=True)
else:
    # If the candidate train_len exceeds available samples, use all available data
    train_fe_final = train_window

print("Data shape for final model training:", train_fe_final.shape)
print(f"Final training window length (actual sample count): {len(train_fe_final)}, from candidate train_len={best_train_len}")

# 3) Define feature columns consistently with CV
feature_cols_final = [
    c for c in train_fe_final.columns
    if c not in ["date_id", "forward_returns", "risk_free_rate"]
]

X_final = train_fe_final[feature_cols_final].values
y_final = train_fe_final["forward_returns"].values

# Record training feature medians for test-set imputation during predict
train_feature_medians_final = train_fe_final[feature_cols_final].median()

lgbm_params_final = dict(
    n_estimators=5000,
    learning_rate=0.001,
    num_leaves=64,
    max_depth=5,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# Train the final LightGBM model
model_final = LGBMRegressor(**lgbm_params_final)
model_final.fit(X_final, y_final)
print("Final LightGBM model trained; feature count:", len(feature_cols_final))
print(f"Final SIGNAL_MULTIPLIER={SIGNAL_MULTIPLIER}")

# ===========================
# Kaggle evaluation interface: predict(test: pl.DataFrame)
# Use exactly the same feature engineering as training, in streaming mode
# ===========================

# Global variable to accumulate test history during evaluation
test_history_df = None

def predict(test: pl.DataFrame) -> float:
    """
    Prediction function called by Kaggle default_inference_server:
    - Input: a Polars DataFrame, usually one row at a time in chronological order.
    - Output: allocation for the latest row, in [0, 2].

    Key points:
    - Use the same preprocess_and_make_features as training.
    - Accumulate test rows in test_history_df so rolling/EWM operations use past data only.
    - Map positions using the best SIGNAL_MULTIPLIER from grid search.
    """
    global test_history_df, model_final, feature_cols_final, train_feature_medians_final, SIGNAL_MULTIPLIER

    # pl → pandas
    test_pd = test.to_pandas()

    # Append to the history, assuming calls arrive in date order
    if test_history_df is None:
        test_history_df = test_pd.copy()
    else:
        test_history_df = pd.concat([test_history_df, test_pd], axis=0, ignore_index=True)

    # test already contains lagged_forward_returns / lagged_risk_free_rate / lagged_market_forward_excess_returns
    # Reuse the shared feature engineering function
    test_fe = preprocess_and_make_features(test_history_df)

    # Add any missing training feature columns using their training medians
    for col in feature_cols_final:
        if col not in test_fe.columns:
            default_val = train_feature_medians_final.get(col, 0.0)
            test_fe[col] = default_val

    X_all = test_fe[feature_cols_final].copy()
    X_all = X_all.replace([np.inf, -np.inf], np.nan)
    X_all = X_all.fillna(train_feature_medians_final)

    # Use the latest row as the current feature vector
    X_last = X_all.iloc[[-1]].values
    pred_ret = float(model_final.predict(X_last)[0])

    # Map to a position allocation using the best SIGNAL_MULTIPLIER
    position_raw = 1.0 + SIGNAL_MULTIPLIER * pred_ret
    alloc = float(np.clip(position_raw, MIN_SIGNAL, MAX_SIGNAL))
    return alloc


In [ ]:
1

# Parameters to evaluate iteratively

- Training data length
- Nonlinear scaling from returns to positions ####################
- Add online learning with intermittent retraining
- Handle test data when using rolling features

# Potentially useful features
- forward_returns: lags 1, 2, 3, 4, 5, 6; rolling means; EMA; etc.
- Explore return-to-position mapping and local validation
- Construct a local 180-day validation period aligned with the leaderboard
- Feature selection

In [ ]:
# ===========================
# Start the Kaggle evaluation service
# ===========================
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))